# Preference Tuning with DPO

**Phase 09** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-09/06-preference-tuning-dpo.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # Running locally — set OPENAI_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("OPENAI_API_KEY")))

## The Problem

You've fine-tuned a model on customer support examples and it writes correct responses. But the tone is wrong. It's too terse, occasionally curt, sometimes overly formal when the customer needs warmth.

You could write a style guide and try to encode it in every training example. But tone is hard to write from scratch. It's easy to say "this response is better than that one." A human rater can look at two responses to the same message and immediately say which they prefer. Writing the "correct" response from a blank slate is much harder.

This is the gap SFT cannot close. SFT teaches the model what to produce. It requires you to have the right answer. DPO teaches the model which of two answe...

## The Concept

### SFT vs. DPO: Two Different Teaching Signals

SFT and DPO use fundamentally different training signals:

```
SFT FORMAT (prompt + single completion)
-----------------------------------------
Each example teaches: "given this input, produce this output"

  prompt: "Summarize this contract clause."
  completion: "The vendor must deliver within 30 days of PO receipt."

DPO FORMAT (prompt + chosen + rejected)
-----------------------------------------
Each example teaches: "given this input, prefer this output over that one"

  prompt: "How do I reset my password?"
  chosen:  "Click 'Forgot pass...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
L06: Preference Tuning with DPO
DPODatasetBuilder: convert human preference annotations to DPO JSONL format.

Usage:
    python main.py                  # run demo with synthetic annotations
    python main.py --input raw.json # convert a real annotation file
    python main.py --stats          # show stats only, no output file
"""

import argparse
import json
import sys
from dataclasses import dataclass, asdict
from typing import Optional
from pathlib import Path


@dataclass
class PreferenceAnnotation:
    """Raw human annotation: two candidates, one preferred."""
    prompt: str
    response_a: str
    response_b: str
    preferred: str  # "A", "B", or "tie"
    annotator_id: Optional[str] = None


@dataclass
class DPOExample:
    """DPO training triplet (prompt, chosen, rejected)."""
    prompt: str
    chosen: str
    rejected: str


class DPODatasetBuilder:
    """
    Convert raw preference annotations to DPO JSONL format.

    Validation rules:
    - Ties are excluded (no clear preference signal)
    - Empty prompts are excluded
    - Identical chosen/rejected are excluded
    - Text length must be within [min_length, max_length]
    """

    def __init__(self, min_length: int = 20, max_length: int = 2000):
        self.min_length = min_length
        self.max_length = max_length
        self.stats = {
            "total": 0,
            "kept": 0,
            "rejected_tie": 0,
            "rejected_empty_prompt": 0,
            "rejected_identical": 0,
            "rejected_length": 0,
        }

    def validate(self, ann: PreferenceAnnotation) -> tuple[bool, str]:
        """
        Validate a single annotation.
        Returns (is_valid, rejection_reason).
        """
        if ann.preferred == "tie":
            return False, "tie"

        if not ann.prompt.strip():
            return False, "empty_prompt"

        if ann.preferred == "A":
            chosen = ann.response_a
            rejected = ann.response_b
        else:
            chosen = ann.response_b
            rejected = ann.response_a

        if chosen.strip() == rejected.strip():
            return False, "identical_responses"

        for label, text in [("chosen", chosen), ("rejected", rejected)]:
            stripped = text.strip()
            if len(stripped) < self.min_length:
                return False, f"too_short_{label}"
            if len(stripped) > self.max_length:
                return False, f"too_long_{label}"

        return True, "ok"

    def convert(self, ann: PreferenceAnnotation) -> Optional[DPOExample]:
        """Convert one annotation to a DPOExample, or None if invalid."""
        valid, reason = self.validate(ann)
        self.stats["total"] += 1

        if not valid:
            if reason == "tie":
                self.stats["rejected_tie"] += 1
            elif reason == "empty_prompt":
                self.stats["rejected_empty_prompt"] += 1
            elif reason == "identical_responses":
                self.stats["rejected_identical"] += 1
            else:  # too_short / too_long
                self.stats["rejected_length"] += 1
            return None

        self.stats["kept"] += 1
        chosen = ann.response_a if ann.preferred == "A" else ann.response_b
        rejected = ann.response_b if ann.preferred == "A" else ann.response_a
        return DPOExample(prompt=ann.prompt, chosen=chosen, rejected=rejected)

    def build(self, annotations: list[PreferenceAnnotation],
              output_path: str) -> dict:
        """
        Convert all annotations, write valid triplets to JSONL.
        Returns stats dict.
        """
        # Reset stats for each build call
        self.stats = {k: 0 for k in self.stats}

        examples = []
        for ann in annotations:
            ex = self.convert(ann)
            if ex:
                examples.append({
                    "prompt": ex.prompt,
                    "chosen": ex.chosen,
                    "rejected": ex.rejected,
                })

        with open(output_path, "w") as f:
            for ex in examples:
                f.write(json.dumps(ex) + "\n")

        keep_rate = (
            self.stats["kept"] / max(self.stats["total"], 1) * 100
        )
        return {
            **self.stats,
            "keep_rate_pct": round(keep_rate, 1),
            "output_path": output_path,
        }

    def print_report(self, result: dict) -> None:
        """Print a human-readable build report."""
        print("\n=== DPO Dataset Build Report ===")
        print(f"  Total annotations:     {result['total']}")
        print(f"  Kept:                  {result['kept']} ({result['keep_rate_pct']}%)")
        print(f"  Rejected - ties:       {result['rejected_tie']}")
        print(f"  Rejected - empty:      {result['rejected_empty_prompt']}")
        print(f"  Rejected - identical:  {result['rejected_identical']}")
        print(f"  Rejected - length:     {result['rejected_length']}")
        print(f"  Output file:           {result['output_path']}")

        if result["keep_rate_pct"] < 60:
            print("\n  WARNING: Keep rate below 60%. Review annotation guidelines.")
            print("  High tie rate suggests annotators see the options as too similar.")
        elif result["keep_rate_pct"] > 95:
            print("\n  INFO: Very high keep rate. Verify quality threshold is set correctly.")
        else:
            print("\n  Quality check: PASS")


def make_demo_annotations() -> list[PreferenceAnnotation]:
    """Synthetic preference annotations for demonstration."""
    return [
        PreferenceAnnotation(
            prompt="How do I reset my password?",
            response_a=(
                "Click 'Forgot password' on the login page. "
                "You'll receive an email within 2 minutes with a reset link. "
                "The link expires after 24 hours."
            ),
            response_b="Use the forgot password link.",
            preferred="A",
            annotator_id="ann_001",
        ),
        PreferenceAnnotation(
            prompt="What is the refund policy?",
            response_a=(
                "We offer full refunds within 30 days of purchase, "
                "no questions asked. After 30 days, store credit only."
            ),
            response_b=(
                "Our refund policy allows returns within 30 days for a full "
                "refund. Items returned after 30 days are eligible for store "
                "credit. Contact support@example.com to initiate a return."
            ),
            preferred="B",
            annotator_id="ann_002",
        ),
        PreferenceAnnotation(
            prompt="Is the product compatible with Windows 11?",
            response_a="Yes.",
            response_b=(
                "Yes, the product is fully compatible with Windows 11 "
                "(versions 21H2 and later). No additional drivers are required."
            ),
            preferred="B",
            annotator_id="ann_001",
        ),
        # Tie - will be excluded
        PreferenceAnnotation(
            prompt="How long does shipping take?",
            response_a="Standard shipping is 5-7 business days.",
            response_b="Shipping typically takes 5-7 business days.",
            preferred="tie",
            annotator_id="ann_003",
        ),
        # Another valid preference
        PreferenceAnnotation(
            prompt="Can I use the product offline?",
            response_a=(
                "Core features work offline. Sync and collaboration features "
                "require an internet connection."
            ),
            response_b="Some features need internet.",
            preferred="A",
            annotator_id="ann_002",
        ),
        # Too short - will be excluded
        PreferenceAnnotation(
            prompt="What formats are supported?",
            response_a="PDF, DOCX, PNG.",
            response_b="Only PDF.",
            preferred="A",
            annotator_id="ann_001",
        ),
        PreferenceAnnotation(
            prompt="How do I cancel my subscription?",
            response_a=(
                "To cancel, go to Account Settings > Subscription > Cancel Plan. "
                "Your access continues until the end of the current billing period."
            ),
            response_b=(
                "You can cancel anytime. Log in, go to settings, find your "
                "subscription, and click cancel. You won't be charged again but "
                "keep access until the period ends."
            ),
            preferred="A",
            annotator_id="ann_003",
        ),
    ]


def load_annotations_from_file(path: str) -> list[PreferenceAnnotation]:
    """
    Load annotations from a JSON file.
    Expected format: list of objects with keys:
      prompt, response_a, response_b, preferred, annotator_id (optional)
    """
    with open(path) as f:
        data = json.load(f)
    annotations = []
    for item in data:
        annotations.append(PreferenceAnnotation(
            prompt=item["prompt"],
            response_a=item["response_a"],
            response_b=item["response_b"],
            preferred=item["preferred"],
            annotator_id=item.get("annotator_id"),
        ))
    return annotations


def analyze_dataset(output_path: str) -> None:
    """Print length statistics for the built dataset."""
    chosen_lengths = []
    rejected_lengths = []

    with open(output_path) as f:
        for line in f:
            ex = json.loads(line)
            chosen_lengths.append(len(ex["chosen"]))
            rejected_lengths.append(len(ex["rejected"]))

    if not chosen_lengths:
        print("No examples in dataset.")
        return

    def stats(values: list[int]) -> str:
        mean = sum(values) / len(values)
        sorted_v = sorted(values)
        median = sorted_v[len(sorted_v) // 2]
        return f"mean={mean:.0f}, median={median}, min={min(values)}, max={max(values)}"

    print("\n=== Dataset Length Analysis ===")
    print(f"  Chosen   (chars): {stats(chosen_lengths)}")
    print(f"  Rejected (chars): {stats(rejected_lengths)}")

    ratio = sum(chosen_lengths) / max(sum(rejected_lengths), 1)
    if ratio > 2.0:
        print(f"\n  WARNING: Chosen responses are {ratio:.1f}x longer than rejected.")
        print("  Model may learn 'be verbose' rather than 'be better'.")
    else:
        print(f"  Length ratio (chosen/rejected): {ratio:.2f} - OK")


def main():
    parser = argparse.ArgumentParser(
        description="Build a DPO training dataset from preference annotations."
    )
    parser.add_argument(
        "--input", "-i",
        help="Path to input JSON annotation file (default: use demo data)",
        default=None,
    )
    parser.add_argument(
        "--output", "-o",
        help="Output JSONL path (default: dpo_dataset.jsonl)",
        default="dpo_dataset.jsonl",
    )
    parser.add_argument(
        "--min-length",
        type=int,
        default=20,
        help="Minimum text length in characters (default: 20)",
    )
    parser.add_argument(
        "--max-length",
        type=int,
        default=2000,
        help="Maximum text length in characters (default: 2000)",
    )
    parser.add_argument(
        "--stats",
        action="store_true",
        help="Also print dataset length statistics after building",
    )
    args = parser.parse_args()

    # Load annotations
    if args.input:
        print(f"Loading annotations from: {args.input}")
        annotations = load_annotations_from_file(args.input)
    else:
        print("Using demo annotations (8 examples, 2 will be filtered)...")
        annotations = make_demo_annotations()

    # Build dataset
    builder = DPODatasetBuilder(
        min_length=args.min_length,
        max_length=args.max_length,
    )
    result = builder.build(annotations, output_path=args.output)
    builder.print_report(result)

    # Optional: show length analysis
    if args.stats and result["kept"] > 0:
        analyze_dataset(args.output)

    # Show a preview of the output
    if result["kept"] > 0:
        print(f"\n=== Preview (first example from {args.output}) ===")
        with open(args.output) as f:
            first = json.loads(f.readline())
        print(f"  Prompt:   {first['prompt'][:80]}...")
        print(f"  Chosen:   {first['chosen'][:80]}...")
        print(f"  Rejected: {first['rejected'][:80]}...")

    return 0 if result["kept"] > 0 else 1

### Demo

In [ ]:
sys.exit(main())

---

## Self-Check

Answer these without running code first:

**1. Which training approach is most appropriate here, and why?**

- A. SFT on the 800 preferred responses only, because you already have correct outputs
- B. DPO using the chosen/rejected pairs, because tone is a subjective quality best taught by preference ranking rather than a target output
- C. Both SFT and DPO simultaneously on the same dataset to maximize signal
- D. Neither: tone cannot be improved through fine-tuning and requires prompt engineering only

**2. What should the team do?**

- A. Lower the minimum length threshold to recover more examples and bring the keep rate above 60%
- B. Proceed with the 1,180 valid triplets; the 59% keep rate is close to healthy and the rejections are correct
- C. Force annotators to choose a winner for every pair, including ties, to recover the 600 rejected examples
- D. Discard the entire dataset and start over because a 41% rejection rate indicates fundamentally flawed annotation guidelines

**3. What most likely caused the accuracy regression?**

- A. The training dataset was too small for DPO to converge
- B. The reference model checkpoint was from a different model family
- C. The beta value is too low, allowing the model to drift too far from the SFT reference
- D. DPO inherently trades task accuracy for alignment and the 19-point drop is expected

**4. What will most likely happen if they run DPO directly on the base model?**

- A. DPO will work correctly because it does not require the model to understand the task first
- B. DPO will fail to run because TRL's DPOTrainer requires an SFT checkpoint as input
- C. DPO on a base model will likely make performance worse because the model needs task capability before preference tuning can refine it
- D. DPO will work but will require 5x more preference pairs to compensate for the missing SFT stage

**5. What problem does this length imbalance create, and how should you address it?**

- A. No problem; longer responses are inherently better and the model should learn to be verbose
- B. The model may learn to maximize response length rather than improve quality; audit whether length correlates with actual quality in your annotations
- C. The imbalance will cause the training to crash because DPO requires equal token counts in chosen and rejected
- D. This is expected and healthy; DPO always produces longer chosen responses as part of alignment

**6. What is the most accurate description of how DPO differs from RLHF in terms of what components are required?**

- A. DPO requires a separate reward model and RL training loop; RLHF does not
- B. DPO eliminates the need for a separate reward model and RL training loop by optimizing preferences directly in a supervised training step
- C. DPO and RLHF are identical in implementation but DPO uses a different loss function name
- D. RLHF is strictly better than DPO for all alignment tasks because it trains a richer reward model

**7. Based on the Evaluate It criteria from this lesson, should you deploy the DPO model?**

- A. Yes, because an 18-point tone improvement clearly outweighs an 8-point accuracy drop
- B. No, because any accuracy regression at all disqualifies a DPO model from deployment
- C. No, because an 8-point accuracy regression exceeds the 5-point threshold; diagnose whether beta is too low or training ran too long before deploying
- D. Yes, but only if you add a disclaimer to users that factual accuracy may have decreased

**8. What risk does this prompt distribution create for the fine-tuned model?**

- A. No risk; more examples of difficult scenarios makes the model better at all scenarios
- B. The model will over-fit to billing and cancellation tone patterns, potentially applying them inappropriately to unrelated scenarios
- C. The DPO loss function automatically rebalances prompt distributions, so this is not an issue
- D. The 75/25 split is ideal because it mirrors the annotation difficulty ratio

_Answers are in `checks.json` in the lesson directory._